# Strategie-Vergleich: EWMA / Bollinger / GARCH Pairtrading

Laedt automatisch die je Basket besten, vom Quick-Tuner gefundenen
Parameter (`data/processed/tuning/best_params_<strategie>.json`, siehe
`quick_tune_ewma.py` / `quick_tune_bollinger.py` / `quick_tune_garch.py`),
fuehrt fuer jede Strategie und jeden getunten Basket EINEN finalen
Backtest auf dem Out-of-Sample-Testzeitraum durch (letzte `TEST_FRACTION`
des verfuegbaren Zeitraums, siehe `tuning_common.split_walkforward_test` --
derselbe Split, den die Quick-Tuner beim Tuning verwendet haben sollten)
und vergleicht die Ergebnisse visuell.

Kosten: `FEE_PRESET` (Standard: `"standard"` = 10.0 bps, Binance-Spot-
Taker-Satz, siehe `Src.config.BINANCE_FEE_PRESETS_BPS`) wird direkt aus
`tuning_common.resolve_fee_bps` uebernommen -- identische Kostenannahme
wie beim Tuning, keine getrennte Konstante hier im Notebook.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))
sys.path.insert(0, str(Path("../..").resolve()))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pairtrading_engine as pte
import bollinger_pairtrading_engine as bpe
import garch_pairtrading_engine as ge
import tuning_common as tc

from Src.config import (
    ASSET_ID_COL, ASSETS_PATH, CLIQUES_DIR, CLOSE_COL,
    MARKET_SYMBOL_COL, OHLCV_DIR, TIMESTAMP_COL,
)
from Src.clustering.helpers import STABLECOIN_SYMBOLS

print("Imports OK")

## 1. Konfiguration

In [ ]:
FEE_PRESET = "standard"        # siehe Src.config.BINANCE_FEE_PRESETS_BPS / tuning_common.resolve_fee_bps
FEE_BPS = tc.resolve_fee_bps(FEE_PRESET)
INITIAL_CAPITAL = tc.FIXED_INITIAL_CAPITAL
TEST_FRACTION = 0.2             # muss mit dem beim Quick-Tuning verwendeten Split uebereinstimmen
CLIQUES_VARIANT = "cliques_raw_12h"  # siehe data/processed/cliques/*.parquet

STRATEGIES = {
    "EWMA":      {"module": pte, "runner": "run_pairtrading_backtest", "strategy_key": "ewma"},
    "Bollinger": {"module": bpe, "runner": "run_bollinger_backtest",   "strategy_key": "bollinger"},
    "GARCH":     {"module": ge,  "runner": "run_garch_backtest",       "strategy_key": "garch"},
}

print(f"Fee-Preset: {FEE_PRESET} -> {FEE_BPS} bps | Startkapital: {INITIAL_CAPITAL:,.0f}")

## 2. Daten laden (Cliquen + OHLCV + Asset-Symbole)

In [ ]:
assets_df = pd.read_parquet(ASSETS_PATH)[["asset_id", "symbol"]].drop_duplicates()
id_to_symbol = dict(zip(assets_df["asset_id"], assets_df["symbol"]))
excluded_ids_stablecoins = set(
    assets_df.loc[assets_df["symbol"].isin(STABLECOIN_SYMBOLS), "asset_id"].tolist()
)

ohlcv_files = sorted(OHLCV_DIR.glob("*.parquet"))
raw = pd.concat([pd.read_parquet(f) for f in ohlcv_files], ignore_index=True)
raw["base_symbol"] = raw[MARKET_SYMBOL_COL].str.replace("USDT$", "", regex=True)
raw = raw.merge(assets_df.rename(columns={"symbol": "base_symbol"}), on="base_symbol", how="left")
raw = raw.dropna(subset=["asset_id"])
raw["asset_id"] = raw["asset_id"].astype(int)
raw[TIMESTAMP_COL] = pd.to_datetime(raw[TIMESTAMP_COL], utc=True)
raw = raw.loc[~raw["asset_id"].isin(excluded_ids_stablecoins)]
ohlcv_df = raw[[TIMESTAMP_COL, ASSET_ID_COL, CLOSE_COL]].copy()

cliques_df_full = pd.read_parquet(CLIQUES_DIR / f"{CLIQUES_VARIANT}.parquet")

data_min = ohlcv_df[TIMESTAMP_COL].min()
data_max = ohlcv_df[TIMESTAMP_COL].max()
print(f"Zeitraum: {data_min.date()} -> {data_max.date()}")
print(f"OHLCV: {ohlcv_df.shape[0]:,} Zeilen | Assets: {ohlcv_df[ASSET_ID_COL].nunique()}")
print(f"Baskets in '{CLIQUES_VARIANT}': {sorted(cliques_df_full['basket'].unique())}")

## 3. Testzeitraum ableiten (identisch zum Quick-Tuning-Split)

In [ ]:
_, _, test_start, test_end = tc.split_walkforward_test(data_min, data_max, test_fraction=TEST_FRACTION)
print(f"Out-of-Sample-Testzeitraum: {test_start.date()} -> {test_end.date()} "
      f"({(test_end - test_start).days} Tage)")

## 4. Beste Parameter je Strategie/Basket laden

In [ ]:
best_params_by_strategy = {name: tc.load_best_params(cfg["strategy_key"]) for name, cfg in STRATEGIES.items()}

available_rows = []
for strat_name, by_basket in best_params_by_strategy.items():
    for basket, params in by_basket.items():
        available_rows.append({
            "strategy": strat_name, "basket": basket,
            "objective": params.get("objective"), "n_trials": params.get("n_trials"),
            "tuned_at": params.get("tuned_at"),
        })
available_df = pd.DataFrame(available_rows)

if available_df.empty:
    print(
        "Keine getunten Parameter gefunden. Vor diesem Notebook einen oder mehrere der "
        "Quick-Tuner laufen lassen:\n"
        "  quick_tune_ewma.tune_ewma_strategy(...)\n"
        "  quick_tune_bollinger.tune_bollinger_strategy(...)\n"
        "  quick_tune_garch.tune_garch_strategy(...)\n"
        "Jeder Aufruf speichert automatisch nach data/processed/tuning/best_params_<strategie>.json."
    )
else:
    display(available_df.sort_values(["basket", "strategy"]))

## 5. Finalen Backtest je (Strategie, Basket) auf dem Testzeitraum ausfuehren

In [ ]:
def _max_holding_candles(params: dict) -> int:
    return max(1, int(round(params["max_holding_hours"] * 60 / 5)))


def run_final_backtest(strategy_name: str, basket: str, params: dict) -> dict:
    """Fuehrt EINEN finalen Backtest fuer (strategy_name, basket) auf dem
    Out-of-Sample-Testzeitraum aus, mit den vom Quick-Tuner gefundenen besten
    Parametern. Gibt trades_df/equity_curve/stats zurueck (wie
    pairtrading_engine.compute_backtest_stats/compute_equity_curve --
    strategieuebergreifend identisch, siehe dortige Modul-Docstrings)."""
    cliques_df = cliques_df_full[cliques_df_full["basket"] == basket]
    max_holding_candles = _max_holding_candles(params)
    common_kwargs = dict(
        cliques_df=cliques_df, ohlcv_df=ohlcv_df, id_to_symbol=id_to_symbol,
        timestamp_column=TIMESTAMP_COL, asset_id_column=ASSET_ID_COL, close_column=CLOSE_COL,
        start_date=test_start, end_date=test_end - pd.Timedelta(days=1),
        max_holding_candles=max_holding_candles, fee_bps=FEE_BPS,
        trade_mode=params["trade_mode"],
    )

    if strategy_name == "EWMA":
        trades_df = pte.run_pairtrading_backtest(
            **common_kwargs,
            decay=params["ewma_decay"], lookback_candles=params["lookback_candles"],
            p_content=params["p_content"], confidence=tc.FIXED_CONFIDENCE,
            exit_lookback_candles=params["exit_lookback_candles"],
            recalib_every=params["recalib_every"],
        )
        max_concurrent = params["max_concurrent_positions"]
    elif strategy_name == "Bollinger":
        trades_df = bpe.run_bollinger_backtest(
            **common_kwargs,
            lookback_candles=params["lookback_candles"], n_std=params["n_std"],
            exit_lookback_candles=params["exit_lookback_candles"],
            hedge_lookback_candles=params["hedge_lookback_candles"],
        )
        max_concurrent = params["max_concurrent_positions"]
    elif strategy_name == "GARCH":
        trades_df = ge.run_garch_backtest(
            **common_kwargs,
            lookback_candles=params["lookback_candles"], p_content=params["p_content"],
            confidence=tc.FIXED_CONFIDENCE, recalib_every=params["recalib_every"],
        )
        max_concurrent = params["max_concurrent_positions"]
    else:
        raise ValueError(f"Unbekannte Strategie: {strategy_name}")

    equity_curve = pte.compute_equity_curve(
        trades_df, initial_capital=INITIAL_CAPITAL, max_concurrent_positions=max_concurrent,
    )
    stats = pte.compute_backtest_stats(trades_df, equity_curve)
    return {"trades_df": trades_df, "equity_curve": equity_curve, "stats": stats}


results: dict[tuple[str, str], dict] = {}
for strat_name, by_basket in best_params_by_strategy.items():
    for basket, params in by_basket.items():
        print(f"Backtest: {strat_name} / {basket} ...")
        try:
            results[(strat_name, basket)] = run_final_backtest(strat_name, basket, params)
        except Exception as exc:
            print(f"  FEHLER bei {strat_name}/{basket}: {exc}")

print(f"\n{len(results)} Backtests abgeschlossen.")

## 6. Kennzahlen-Tabelle

In [ ]:
stats_rows = []
for (strat_name, basket), res in results.items():
    stats_rows.append({"strategy": strat_name, "basket": basket, **res["stats"]})
stats_df = pd.DataFrame(stats_rows).sort_values(["basket", "strategy"]).reset_index(drop=True)
display(stats_df)

## 7. Plots

In [ ]:
STRATEGY_COLORS = {"EWMA": "#2a78d6", "Bollinger": "#eb6834", "GARCH": "#1baf7a"}


def plot_equity_curves_by_basket(results: dict, *, show: bool = True):
    """Ein Subplot je Basket, darin je eine Equity-Kurve pro Strategie --
    direkter visueller Vergleich, wie stark sich die drei Signalarten auf
    denselben Cliquen unterscheiden."""
    baskets = sorted({b for _, b in results})
    fig, axes = plt.subplots(len(baskets), 1, figsize=(11, 4 * len(baskets)), squeeze=False)
    axes = axes[:, 0]

    for ax, basket in zip(axes, baskets):
        for strat_name in STRATEGIES:
            res = results.get((strat_name, basket))
            if res is None or res["equity_curve"].empty:
                continue
            curve = res["equity_curve"]
            ax.plot(curve.index, curve.values, color=STRATEGY_COLORS[strat_name],
                     linewidth=1.3, label=strat_name)
        ax.axhline(INITIAL_CAPITAL, color="#898781", linewidth=1, linestyle="--", label="Startkapital")
        ax.set_title(f"Equity-Kurve -- {basket}")
        ax.set_xlabel("Zeit")
        ax.set_ylabel("Kapital")
        ax.legend(frameon=False, fontsize=8)
        ax.grid(True, alpha=0.25)

    fig.autofmt_xdate()
    fig.tight_layout()
    if show:
        plt.show()
    return fig


def plot_metric_bars(stats_df: pd.DataFrame, metric: str, *, ylabel: str, show: bool = True):
    """Gruppierte Balken: x=Basket, Balkengruppe=Strategie, Hoehe=metric."""
    baskets = sorted(stats_df["basket"].unique())
    strat_names = list(STRATEGIES.keys())
    x = np.arange(len(baskets))
    width = 0.8 / len(strat_names)

    fig, ax = plt.subplots(figsize=(9, 4.5))
    for i, strat_name in enumerate(strat_names):
        values = []
        for basket in baskets:
            row = stats_df.loc[(stats_df["basket"] == basket) & (stats_df["strategy"] == strat_name), metric]
            values.append(row.iloc[0] if len(row) else np.nan)
        ax.bar(x + i * width, values, width=width, label=strat_name, color=STRATEGY_COLORS[strat_name])

    ax.axhline(0, color="#898781", linewidth=0.8)
    ax.set_xticks(x + width * (len(strat_names) - 1) / 2)
    ax.set_xticklabels(baskets, rotation=0)
    ax.set_ylabel(ylabel)
    ax.set_title(f"{ylabel} je Basket/Strategie")
    ax.legend(frameon=False)
    ax.grid(True, axis="y", alpha=0.25)
    fig.tight_layout()
    if show:
        plt.show()
    return fig


def plot_trade_return_distributions(results: dict, *, show: bool = True):
    """Ein Subplot je Basket: Histogramm der Trade-net_return-Verteilung je
    Strategie -- zeigt, ob eine Strategie viele kleine Gewinne/Verluste oder
    wenige grosse Ausreisser produziert."""
    baskets = sorted({b for _, b in results})
    fig, axes = plt.subplots(1, len(baskets), figsize=(6 * len(baskets), 4), squeeze=False)
    axes = axes[0]

    for ax, basket in zip(axes, baskets):
        for strat_name in STRATEGIES:
            res = results.get((strat_name, basket))
            if res is None or res["trades_df"].empty:
                continue
            ax.hist(res["trades_df"]["net_return"] * 100, bins=40, alpha=0.5,
                    color=STRATEGY_COLORS[strat_name], label=strat_name)
        ax.axvline(0, color="#898781", linewidth=0.8)
        ax.set_title(f"Trade-Renditen -- {basket}")
        ax.set_xlabel("Net Return (%)")
        ax.set_ylabel("Anzahl Trades")
        ax.legend(frameon=False, fontsize=8)
        ax.grid(True, alpha=0.25)

    fig.tight_layout()
    if show:
        plt.show()
    return fig


def plot_drawdown_curves(results: dict, *, show: bool = True):
    """Ein Subplot je Basket: laufender Drawdown (%) je Strategie ueber die
    Zeit -- ergaenzt die Equity-Kurve um die Risikosicht."""
    baskets = sorted({b for _, b in results})
    fig, axes = plt.subplots(len(baskets), 1, figsize=(11, 3.5 * len(baskets)), squeeze=False)
    axes = axes[:, 0]

    for ax, basket in zip(axes, baskets):
        for strat_name in STRATEGIES:
            res = results.get((strat_name, basket))
            if res is None or res["equity_curve"].empty:
                continue
            curve = res["equity_curve"]
            drawdown = (curve / curve.cummax() - 1.0) * 100
            ax.plot(drawdown.index, drawdown.values, color=STRATEGY_COLORS[strat_name],
                     linewidth=1.2, label=strat_name)
        ax.set_title(f"Drawdown -- {basket}")
        ax.set_xlabel("Zeit")
        ax.set_ylabel("Drawdown (%)")
        ax.legend(frameon=False, fontsize=8)
        ax.grid(True, alpha=0.25)

    fig.autofmt_xdate()
    fig.tight_layout()
    if show:
        plt.show()
    return fig


if results:
    plot_equity_curves_by_basket(results)
else:
    print("Keine Ergebnisse zum Plotten -- siehe Abschnitt 4/5.")

In [ ]:
if not stats_df.empty:
    plot_metric_bars(stats_df, "sharpe", ylabel="Sharpe Ratio")

In [ ]:
if not stats_df.empty:
    plot_metric_bars(stats_df, "total_return", ylabel="Total Return")

In [ ]:
if not stats_df.empty:
    plot_metric_bars(stats_df, "max_drawdown", ylabel="Max Drawdown")

In [ ]:
if results:
    plot_trade_return_distributions(results)

In [ ]:
if results:
    plot_drawdown_curves(results)

## 8. Bestes Setup je Basket (nach Sharpe)

In [ ]:
if not stats_df.empty:
    best_by_basket = stats_df.loc[stats_df.groupby("basket")["sharpe"].idxmax()]
    display(best_by_basket[["basket", "strategy", "sharpe", "total_return", "win_rate", "max_drawdown", "n_trades"]])